* XGBoost 버전 확인

In [ ]:
# xgboost 라이브러리(파이썬 패키지)를 임포트
# XGBoost는 Extreme Gradient Boosting의 약자로, GBM 기반의 알고리즘이지만
# 병렬 학습, 정규화(Regularization), 조기 중단(Early Stopping) 등을 지원하여
# 분류 및 회귀 문제에서 매우 뛰어난 성능을 보이는 부스팅 계열의 머신러닝 라이브러리.
import xgboost

# 현재 설치된 xgboost 버전 정보를 출력
# (버전에 따라 사용 가능한 파라미터 및 API가 일부 다를 수 있으므로 확인 필요)
print(xgboost.__version__)

### 파이썬 래퍼 XGBoost 적용 – 위스콘신 유방암 예측

In [ ]:
# 파이썬 래퍼 XGBoost 모듈을 xgb 별칭으로 임포트 (네이티브 XGBoost API 사용 시 활용)
import xgboost as xgb
# 피처 중요도를 시각화하기 위한 plot_importance 함수 임포트
from xgboost import plot_importance
# 데이터 분석/처리에 사용하는 pandas, numpy 임포트
import pandas as pd
import numpy as np
# 사이킷런에서 제공하는 위스콘신 유방암 데이터셋 로더 임포트
from sklearn.datasets import load_breast_cancer
# 학습/테스트 데이터 분리 함수 임포트
from sklearn.model_selection import train_test_split
# 코드 실행 시 발생하는 경고(warning) 메시지를 무시하도록 설정
import warnings
warnings.filterwarnings('ignore')

# 위스콘신 유방암 데이터셋 로드 (Bunch 객체 형태로 반환됨)
dataset = load_breast_cancer()
# 피처(독립 변수) 데이터: 30개의 종양 관련 측정값
X_features= dataset.data
# 레이블(종속 변수) 데이터: 0(악성, malignant) / 1(양성, benign)
y_label = dataset.target

# 피처 데이터를 DataFrame으로 변환하고 컬럼명을 dataset.feature_names로 지정
cancer_df = pd.DataFrame(data=X_features, columns=dataset.feature_names)
# DataFrame에 'target' 컬럼을 추가해 레이블 값을 함께 보관 (분석 편의를 위함)
cancer_df['target']= y_label
# 상위 3개 샘플을 출력해 데이터의 구조와 값을 확인
cancer_df.head(3)


In [ ]:
# 레이블의 클래스 이름을 출력: ['malignant'(악성=0), 'benign'(양성=1)]
print(dataset.target_names)
# 'target' 컬럼의 클래스별 샘플 수 분포 확인
# (이진 분류 문제에서 클래스 불균형 여부를 점검하는 용도)
print(cancer_df['target'].value_counts())

In [ ]:
# cancer_df에서 feature용 DataFrame과 Label용 Series 객체 추출
# 맨 마지막 칼럼이 Label임. Feature용 DataFrame은 cancer_df의 첫번째 칼럼에서 맨 마지막 두번째 칼럼까지를 :-1 슬라이싱으로 추출.
# iloc[:, :-1]: 모든 행, 마지막 컬럼('target')을 제외한 모든 컬럼을 피처로 선택
X_features = cancer_df.iloc[:, :-1]
# iloc[:, -1]: 모든 행, 가장 마지막 컬럼('target')을 레이블로 선택
y_label = cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
# random_state=156: 동일한 분할 결과 재현을 위한 시드 고정값
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label,
                                         test_size=0.2, random_state=156 )

# 위에서 만든 X_train, y_train을 다시 쪼개서 90%는 학습과 10%는 검증용 데이터로 분리
# 검증(validation) 데이터는 학습 도중 모델 성능 모니터링 및 조기 중단(early stopping)에 사용
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )
# 학습/테스트 데이터 분할 결과 shape 출력 (전체 데이터 기준 80:20)
print(X_train.shape , X_test.shape)
# 학습용 데이터 중 학습/검증 데이터 분할 결과 shape 출력 (X_train 기준 90:10)
print(X_tr.shape, X_val.shape)

In [ ]:
# 만약 구버전 XGBoost에서 DataFrame으로 DMatrix 생성이 안될 경우 X_train.values로 넘파이 변환.
# 학습, 검증, 테스트용 DMatrix를 생성.
# DMatrix는 XGBoost 전용 자료구조로, 내부적으로 데이터를 효율적으로 처리할 수 있도록 최적화된 형태.
# 파이썬 래퍼 XGBoost(xgb.train)를 사용할 때는 반드시 DMatrix 형태로 변환해서 입력해야 함.
# data: 피처 데이터, label: 타겟(레이블) 데이터

# 학습용 DMatrix 생성
dtr = xgb.DMatrix(data=X_tr, label=y_tr)
# 검증용 DMatrix 생성 (early stopping 평가에 사용)
dval = xgb.DMatrix(data=X_val, label=y_val)
# 테스트용 DMatrix 생성 (최종 모델 성능 평가에 사용)
dtest = xgb.DMatrix(data=X_test , label=y_test)

In [ ]:
# XGBoost 학습에 사용할 하이퍼 파라미터를 딕셔너리 형태로 정의
params = { 'max_depth':3,                  # 트리의 최대 깊이. 깊을수록 모델 복잡도 ↑, 과적합 위험 ↑ (보통 3~10)
          'eta': 0.05,                     # 학습률(learning rate). 각 트리가 결과에 기여하는 비율 (작을수록 안정적이지만 학습 속도 ↓)
          'objective':'binary:logistic',   # 학습 목적 함수. 이진 분류 + 로지스틱 회귀 (확률값 출력)
          'eval_metric':'logloss'          # 검증 시 사용할 평가지표. logloss(작을수록 좋음)
         }
# 부스팅 라운드 수(약한 학습기 트리의 개수). 최대 400개의 트리를 순차적으로 생성하며 학습 진행
num_rounds = 400

In [ ]:
# 학습 데이터 셋은 'train' 또는 평가 데이터 셋은 'eval' 로 명기합니다. 
# (DMatrix, '이름') 형식의 튜플 리스트로 전달하면, 학습 중 각 데이터셋의 평가지표가 출력됨
eval_list = [(dtr,'train'),(dval,'eval')] # 또는 eval_list = [(dval,'eval')] 만 명기해도 무방. 

# 하이퍼 파라미터와 early stopping 파라미터를 train( ) 함수의 파라미터로 전달
# params: 위에서 정의한 하이퍼 파라미터 딕셔너리
# dtrain: 학습용 DMatrix
# num_boost_round: 최대 부스팅 반복 횟수 (= 트리 개수)
# early_stopping_rounds=50: 검증 데이터(eval)의 성능이 50회 연속 개선되지 않으면 학습 조기 종료
# evals: 평가 데이터셋 리스트 (학습 중 성능을 모니터링)
xgb_model = xgb.train(params = params , dtrain=dtr , num_boost_round=num_rounds , \
                      early_stopping_rounds=50, evals=eval_list )

In [ ]:
# 학습된 XGBoost 모델을 사용해 테스트 데이터(dtest)에 대한 예측 확률값(0~1 사이) 산출
# objective='binary:logistic' 이므로 양성 클래스(1)에 속할 확률을 반환
pred_probs = xgb_model.predict(dtest)
print('predict( ) 수행 결과값을 10개만 표시, 예측 확률 값으로 표시됨')
# 소수점 3자리로 반올림하여 상위 10개의 예측 확률을 확인
print(np.round(pred_probs[:10],3))

# 예측 확률이 0.5 보다 크면 1 , 그렇지 않으면 0 으로 예측값 결정하여 List 객체인 preds에 저장 
# 리스트 컴프리헨션을 이용한 임계값(threshold) 기반 이진 분류 변환
preds = [ 1 if x > 0.5 else 0 for x in pred_probs ]
# 변환된 예측 클래스 값 상위 10개 확인
print('예측값 10개만 표시:',preds[:10])

In [ ]:
# 분류 성능 평가에 사용할 사이킷런 메트릭 함수들 임포트
from sklearn.metrics import confusion_matrix, accuracy_score   # 오차행렬, 정확도
from sklearn.metrics import precision_score, recall_score      # 정밀도, 재현율
from sklearn.metrics import f1_score, roc_auc_score            # F1 스코어, ROC-AUC

# 분류 모델 성능 평가 결과를 한 번에 출력해주는 사용자 정의 함수
# y_test: 실제 정답 레이블
# pred: 예측 클래스(0/1)
# pred_proba: 양성 클래스 예측 확률 (ROC-AUC 계산에 필요)
def get_clf_eval(y_test, pred=None, pred_proba=None):
    # 오차행렬: [[TN, FP], [FN, TP]] 형태로 반환
    confusion = confusion_matrix( y_test, pred)
    # 정확도: 전체 중 올바르게 예측한 비율
    accuracy = accuracy_score(y_test , pred)
    # 정밀도: 양성으로 예측한 것 중 실제 양성 비율 (TP / (TP+FP))
    precision = precision_score(y_test , pred)
    # 재현율: 실제 양성 중 양성으로 예측한 비율 (TP / (TP+FN))
    recall = recall_score(y_test , pred)
    # F1 스코어: 정밀도와 재현율의 조화 평균 (불균형 데이터에 적합한 지표)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 
    # ROC-AUC: 임계값 변화에 따른 분류 성능을 나타내는 종합 지표 (1에 가까울수록 우수)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    # 평가 지표 5종을 소수점 4자리까지 한 줄로 출력
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 위에서 정의한 get_clf_eval() 함수에 실제 정답(y_test), 예측 클래스(preds),
# 예측 확률(pred_probs)을 전달해 파이썬 래퍼 XGBoost의 분류 성능 평가 결과를 출력
get_clf_eval(y_test , preds, pred_probs)

In [ ]:
# 시각화 라이브러리 matplotlib 임포트
import matplotlib.pyplot as plt
# 주피터 노트북 내에서 그래프를 인라인으로 출력하기 위한 매직 커맨드
%matplotlib inline

# 가로 10, 세로 12 크기의 Figure(전체 그림)와 Axes(서브 플롯) 생성
fig, ax = plt.subplots(figsize=(10, 12))
# 학습된 XGBoost 모델(xgb_model)의 피처 중요도(Feature Importance)를 막대 그래프로 시각화
# (기본적으로 F-score: 각 피처가 트리 분할에 사용된 횟수를 기준으로 정렬)
plot_importance(xgb_model, ax=ax)
# 시각화한 피처 중요도 그래프를 TIFF 이미지 파일로 저장 (해상도 300 dpi, 여백 자동 조정)
plt.savefig('p239_xgb_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')

### 사이킷런 래퍼 XGBoost의 개요 및 적용

In [ ]:
# 사이킷런 래퍼 XGBoost 클래스인 XGBClassifier 임포
# (사이킷런 fit/predict 인터페이스와 호환되므로 GridSearchCV, Pipeline 등과 함께 사용 가능)
from xgboost import XGBClassifier

# XGBClassifier 객체 생성
# n_estimators=400: 생성할 트리 개수 (파이썬 래퍼의 num_boost_round에 해당)
# learning_rate=0.1: 학습률 (파이썬 래퍼의 eta에 해당)
# max_depth=3: 트리 최대 깊이
xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)
# 학습 데이터(X_train, y_train)로 모델 학습 수행 (사이킷런과 동일한 fit 메서드 사용)
xgb_wrapper.fit(X_train, y_train)
# 테스트 데이터에 대한 클래스 예측 (0 또는 1)
w_preds = xgb_wrapper.predict(X_test)
# 테스트 데이터에 대한 클래스별 확률 예측 후, 양성(1) 클래스 확률만 슬라이싱
# predict_proba 결과의 shape는 (n_samples, 2): [클래스 0 확률, 클래스 1 확률]
w_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# 사이킷런 래퍼 XGBClassifier 모델의 분류 성능 평가 결과 출력
get_clf_eval(y_test , w_preds, w_pred_proba)

In [ ]:
from xgboost import XGBClassifier

# XGBClassifier 다시 생성 (이전과 동일한 하이퍼 파라미터)
xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)
# early_stopping에 사용할 평가 데이터셋 리스트 (튜플 형태: (X, y))
# 주의: 일반적으로는 별도 검증 데이터(X_val, y_val)를 사용해야 하나, 여기서는 학습 예시를 위해 테스트 데이터를 사용
evals = [(X_test, y_test)]
# 사이킷런 래퍼 XGBoost의 fit() 메서드에 early stopping 옵션 전달
# early_stopping_rounds=100: 평가 지표가 100회 연속 개선되지 않으면 학습 조기 종료
# eval_metric='logloss': 평가에 사용할 지표
# eval_set=evals: 학습 중 평가에 사용할 데이터셋
# verbose=True: 학습 진행 상황(라운드별 logloss)을 콘솔에 출력
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=100, eval_metric="logloss", 
                eval_set=evals, verbose=True)

# early stopping이 적용된 모델의 예측 클래스
ws100_preds = xgb_wrapper.predict(X_test)
# early stopping이 적용된 모델의 양성 클래스 예측 확률
ws100_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# early_stopping_rounds=100을 적용한 모델의 분류 성능 평가 결과 출력
get_clf_eval(y_test , ws100_preds, ws100_pred_proba)

In [ ]:
# early_stopping_rounds를 10으로 설정하고 재 학습. 
# 조기 중단 라운드를 너무 작게 설정하면 충분히 학습되지 않은 채 종료되어 성능 저하가 발생할 수 있음
# (early_stopping_rounds 값에 따른 성능 변화를 비교하기 위한 예제)
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=10, 
                eval_metric="logloss", eval_set=evals,verbose=True)

# early_stopping_rounds=10 적용 모델의 테스트 예측 클래스
ws10_preds = xgb_wrapper.predict(X_test)
# early_stopping_rounds=10 적용 모델의 양성 클래스 예측 확률
ws10_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]
# early_stopping_rounds=10 적용 모델의 분류 성능 평가 결과 출력
get_clf_eval(y_test , ws10_preds, ws10_pred_proba)

In [ ]:
# 피처 중요도 시각화 함수 임포트
from xgboost import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

# 가로 10, 세로 12 크기의 Figure와 Axes 생성
fig, ax = plt.subplots(figsize=(10, 12))
# 사이킷런 래퍼 클래스를 입력해도 무방. 
# plot_importance는 파이썬 래퍼(xgb.Booster) 객체와 사이킷런 래퍼(XGBClassifier) 객체 모두 지원
# 각 피처가 트리 분할에 사용된 빈도(F-score) 순으로 정렬된 막대 그래프 출력
plot_importance(xgb_wrapper, ax=ax)
